In [0]:
from datetime import datetime

SILVER = "workspace.olist_silver"
GOLD = "workspace.olist_gold"

In [0]:
# Cada item: (nome, tabela, nível, descrição, consulta que devolve o número de violações)
verificacoes = [
    # --- Chaves únicas ---
    ("orders_chave_unica", "silver.orders", "erro", "order_id não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM {SILVER}.orders"),
    ("customers_chave_unica", "silver.customers", "erro", "customer_id não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM {SILVER}.customers"),
    ("order_items_chave_unica", "silver.order_items", "erro", "order_id + order_item_id não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT concat(order_id, '-', order_item_id)) FROM {SILVER}.order_items"),
    ("products_chave_unica", "silver.products", "erro", "product_id não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM {SILVER}.products"),
    ("sellers_chave_unica", "silver.sellers", "erro", "seller_id não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT seller_id) FROM {SILVER}.sellers"),
    ("reviews_uma_por_pedido", "silver.order_reviews", "erro", "no máximo uma avaliação por pedido",
     f"SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM {SILVER}.order_reviews"),
    ("dim_date_chave_unica", "gold.dim_date", "erro", "full_date não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT full_date) FROM {GOLD}.dim_date"),
    ("agg_payments_chave_unica", "gold.agg_order_payments", "erro", "uma linha por pedido",
     f"SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM {GOLD}.agg_order_payments"),
    ("fact_chave_unica", "gold.fact_order_items", "erro", "order_id + order_item_id não se repete",
     f"SELECT COUNT(*) - COUNT(DISTINCT concat(order_id, '-', order_item_id)) FROM {GOLD}.fact_order_items"),

    # --- Campos obrigatórios ---
    ("orders_cliente_nao_nulo", "silver.orders", "erro", "customer_id preenchido",
     f"SELECT COUNT(*) FROM {SILVER}.orders WHERE customer_id IS NULL"),
    ("orders_compra_nao_nula", "silver.orders", "erro", "data da compra preenchida",
     f"SELECT COUNT(*) FROM {SILVER}.orders WHERE order_purchase_timestamp IS NULL"),
    ("order_items_produto_vendedor", "silver.order_items", "erro", "product_id e seller_id preenchidos",
     f"SELECT COUNT(*) FROM {SILVER}.order_items WHERE product_id IS NULL OR seller_id IS NULL"),

    # --- Valores válidos ---
    ("reviews_nota_1_a_5", "silver.order_reviews", "erro", "nota entre 1 e 5",
     f"SELECT COUNT(*) FROM {SILVER}.order_reviews WHERE review_score NOT BETWEEN 1 AND 5"),
    ("order_items_valores_positivos", "silver.order_items", "erro", "preço e frete não negativos",
     f"SELECT COUNT(*) FROM {SILVER}.order_items WHERE price < 0 OR freight_value < 0"),
    ("payments_valores_validos", "silver.order_payments", "erro", "valor não negativo e parcelas >= 1",
     f"SELECT COUNT(*) FROM {SILVER}.order_payments WHERE payment_value < 0 OR payment_installments < 1"),
    ("geolocation_dentro_do_brasil", "silver.geolocation", "erro", "coordenadas dentro do Brasil",
     f"""SELECT COUNT(*) FROM {SILVER}.geolocation
         WHERE geolocation_lat NOT BETWEEN -34 AND 6 OR geolocation_lng NOT BETWEEN -74 AND -34"""),

    # --- Coerência entre datas ---
    ("entrega_apos_compra", "silver.orders", "erro", "entrega ao cliente não ocorre antes da compra",
     f"""SELECT COUNT(*) FROM {SILVER}.orders
         WHERE order_delivered_customer_date < order_purchase_timestamp"""),
    ("aprovacao_apos_compra", "silver.orders", "aviso", "aprovação não ocorre antes da compra",
     f"""SELECT COUNT(*) FROM {SILVER}.orders
         WHERE order_approved_at < order_purchase_timestamp"""),
    ("entrega_apos_postagem", "silver.orders", "aviso", "entrega ao cliente não ocorre antes da postagem",
     f"""SELECT COUNT(*) FROM {SILVER}.orders
         WHERE order_delivered_customer_date < order_delivered_carrier_date"""),

    # --- Integridade entre a fato e as dimensões ---
    ("fact_produto_existe", "gold.fact_order_items", "erro", "todo product_id existe em dim_products",
     f"""SELECT COUNT(*) FROM {GOLD}.fact_order_items f
         LEFT JOIN {GOLD}.dim_products d ON f.product_id = d.product_id WHERE d.product_id IS NULL"""),
    ("fact_vendedor_existe", "gold.fact_order_items", "erro", "todo seller_id existe em dim_sellers",
     f"""SELECT COUNT(*) FROM {GOLD}.fact_order_items f
         LEFT JOIN {GOLD}.dim_sellers d ON f.seller_id = d.seller_id WHERE d.seller_id IS NULL"""),
    ("fact_cliente_existe", "gold.fact_order_items", "erro", "todo customer_id existe em dim_customers",
     f"""SELECT COUNT(*) FROM {GOLD}.fact_order_items f
         LEFT JOIN {GOLD}.dim_customers d ON f.customer_id = d.customer_id WHERE d.customer_id IS NULL"""),
    ("fact_data_existe", "gold.fact_order_items", "erro", "toda data de compra existe em dim_date",
     f"""SELECT COUNT(*) FROM {GOLD}.fact_order_items f
         LEFT JOIN {GOLD}.dim_date d ON f.purchase_date = d.full_date WHERE d.full_date IS NULL"""),

    # --- Reconciliação entre camadas ---
    ("fact_mesmas_linhas_da_prata", "gold.fact_order_items", "erro", "a fato tem as mesmas linhas de order_items",
     f"""SELECT CAST(
           (SELECT COUNT(*) FROM {GOLD}.fact_order_items) <> (SELECT COUNT(*) FROM {SILVER}.order_items)
         AS INT)"""),
    ("fact_mesma_soma_de_vendas", "gold.fact_order_items", "erro", "soma de price igual à da prata",
     f"""SELECT CAST(ABS(
           (SELECT SUM(price) FROM {GOLD}.fact_order_items) - (SELECT SUM(price) FROM {SILVER}.order_items)
         ) > 0.01 AS INT)"""),
    ("payments_mesma_soma", "gold.agg_order_payments", "erro", "total pago igual à soma dos pagamentos da prata",
     f"""SELECT CAST(ABS(
           (SELECT SUM(total_paid) FROM {GOLD}.agg_order_payments) - (SELECT SUM(payment_value) FROM {SILVER}.order_payments)
         ) > 0.01 AS INT)"""),
]

print(f"{len(verificacoes)} verificações definidas")

26 verificações definidas


In [0]:
momento = datetime.now()
resultados = []

for nome, tabela, nivel, descricao, consulta in verificacoes:
    violacoes = int(spark.sql(consulta).collect()[0][0])
    status = "PASS" if violacoes == 0 else "FAIL"
    resultados.append((momento, nome, tabela, nivel, descricao, violacoes, status))
    icone = "✅" if status == "PASS" else ("❌" if nivel == "erro" else "⚠️")
    print(f"{icone} {nome}: {violacoes} violações")

df_resultados = spark.createDataFrame(
    resultados,
    "run_ts timestamp, check_name string, table_name string, level string, "
    "description string, violations long, status string",
)
df_resultados.write.mode("append").saveAsTable(f"{GOLD}.data_quality_results")

✅ orders_chave_unica: 0 violações
✅ customers_chave_unica: 0 violações
✅ order_items_chave_unica: 0 violações
✅ products_chave_unica: 0 violações
✅ sellers_chave_unica: 0 violações
✅ reviews_uma_por_pedido: 0 violações
✅ dim_date_chave_unica: 0 violações
✅ agg_payments_chave_unica: 0 violações
✅ fact_chave_unica: 0 violações
✅ orders_cliente_nao_nulo: 0 violações
✅ orders_compra_nao_nula: 0 violações
✅ order_items_produto_vendedor: 0 violações
✅ reviews_nota_1_a_5: 0 violações
✅ order_items_valores_positivos: 0 violações
✅ payments_valores_validos: 0 violações
✅ geolocation_dentro_do_brasil: 0 violações
✅ entrega_apos_compra: 0 violações
✅ aprovacao_apos_compra: 0 violações
⚠️ entrega_apos_postagem: 23 violações
✅ fact_produto_existe: 0 violações
✅ fact_vendedor_existe: 0 violações
✅ fact_cliente_existe: 0 violações
✅ fact_data_existe: 0 violações
✅ fact_mesmas_linhas_da_prata: 0 violações
✅ fact_mesma_soma_de_vendas: 0 violações
✅ payments_mesma_soma: 0 violações


In [0]:
erros = [r for r in resultados if r[6] == "FAIL" and r[3] == "erro"]
avisos = [r for r in resultados if r[6] == "FAIL" and r[3] == "aviso"]
aprovadas = [r for r in resultados if r[6] == "PASS"]

print(f"Aprovadas: {len(aprovadas)} | Avisos: {len(avisos)} | Erros: {len(erros)}")

if erros:
    raise Exception(f"{len(erros)} verificação(ões) com erro: {[r[1] for r in erros]}")

Aprovadas: 25 | Avisos: 1 | Erros: 0


In [0]:
DATA_CORTE = "2018-08-01"

spark.sql(f"DROP TABLE IF EXISTS {GOLD}.fact_order_items_inc")

spark.sql(f"""
CREATE TABLE {GOLD}.fact_order_items_inc AS
SELECT * FROM {GOLD}.fact_order_items
WHERE purchase_date < DATE'{DATA_CORTE}'
""")

inicial = spark.table(f"{GOLD}.fact_order_items_inc").count()
print(f"Carga inicial: {inicial} linhas")

Carga inicial: 105401 linhas


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW lote_novo AS
SELECT * FROM {GOLD}.fact_order_items
WHERE purchase_date >= DATE'{DATA_CORTE}'
""")

novos = spark.table("lote_novo").count()
print(f"Lote novo: {novos} linhas")

Lote novo: 7249 linhas


In [0]:
resultado_merge = spark.sql(f"""
MERGE INTO {GOLD}.fact_order_items_inc AS destino
USING lote_novo AS origem
ON destino.order_id = origem.order_id
   AND destino.order_item_id = origem.order_item_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(resultado_merge)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
7249,0,0,7249


In [0]:
repeticao = spark.sql(f"""
MERGE INTO {GOLD}.fact_order_items_inc AS destino
USING lote_novo AS origem
ON destino.order_id = origem.order_id
   AND destino.order_item_id = origem.order_item_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(repeticao)
print("Linhas no destino depois da repetição:", spark.table(f"{GOLD}.fact_order_items_inc").count())

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
7249,7249,0,0


Linhas no destino depois da repetição: 112650
